# 00 — Dataset audit: duplicate records and acquisition parameters

This notebook is run on the **original public dataset**, before any preprocessing. It:

1. builds an inventory of the EDF files (group, subject, condition);
2. checks the acquisition parameters stored in each EDF header;
3. identifies duplicate records at two levels with SHA-256:
   - **file level**: hash of the complete EDF file (byte-identical files);
   - **signal level**: hash of the 19 EEG channels only, ignoring header and metadata
     (the same physiological recording stored under different identifiers);
4. uses the recording timestamps in the EDF headers to attribute a record shared by
   several identifiers to its most likely owner;
5. writes the list of records excluded from the analysis.

The exclusion list produced here is the one applied in `01_preprocessing.ipynb`
(`EXCLUDED_RECORDS`).

**Input:** folder with the original EDF files.
**Outputs (in `OUTPUT_DIR`):** `file_inventory.csv`, `duplicate_audit.csv`,
`shared_record_attribution.csv`, `excluded_records.csv`.

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
# Skip this cell when running locally with requirements.txt installed.
import sys
if 'google.colab' in sys.modules:
    !pip install -q mne==1.13.2
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import hashlib
import os
import re
from datetime import timedelta
from pathlib import Path

import mne
import numpy as np
import pandas as pd

mne.set_log_level('ERROR')
print(f'MNE {mne.__version__} | NumPy {np.__version__} | pandas {pd.__version__}')

## Configuration

In [ ]:
DATA_DIR = 'path/to/original_edf_files'   # folder with the original EDF files
OUTPUT_DIR = 'path/to/audit_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONDITIONS = ['EC', 'EO', 'TASK']

# The 19 EEG channels of the 10-20 system, as named in the EDF files.
EEG_CHANNELS = [
    'EEG Fp1-LE', 'EEG Fp2-LE',
    'EEG F7-LE', 'EEG F3-LE', 'EEG Fz-LE', 'EEG F4-LE', 'EEG F8-LE',
    'EEG T3-LE', 'EEG C3-LE', 'EEG Cz-LE', 'EEG C4-LE', 'EEG T4-LE',
    'EEG T5-LE', 'EEG P3-LE', 'EEG Pz-LE', 'EEG P4-LE', 'EEG T6-LE',
    'EEG O1-LE', 'EEG O2-LE',
]

## 1. File inventory

In [ ]:
def parse_filename(path):
    # File names follow '<H|MDD> S<number> <EC|EO|TASK>.edf', sometimes with extra spaces.
    base = Path(path).name.upper().replace('.EDF', '').strip()
    condition = next((c for c in CONDITIONS if base.endswith(c)), None)
    if base.startswith('MDD'):
        group = 'MDD'
    elif base.startswith('H'):
        group = 'HC'
    else:
        group = None
    m = re.search(r'S\s*(\d+)', base)
    number = int(m.group(1)) if m else None
    return group, number, condition


rows = []
for fp in sorted(Path(DATA_DIR).glob('*.edf')):
    group, number, condition = parse_filename(fp)
    rows.append({'file': fp.name, 'group': group, 'subject': number,
                 'subject_id': f'{group}_{number}' if group and number is not None else None,
                 'condition': condition})
inventory = pd.DataFrame(rows)

unparsed = inventory[inventory[['group', 'subject', 'condition']].isna().any(axis=1)]
if len(unparsed):
    print('Files with unrecognized names:')
    print(unparsed.to_string(index=False))

collisions = inventory.dropna().groupby(['subject_id', 'condition']).filter(lambda g: len(g) > 1)
print('ID collisions (same subject and condition in more than one file):',
      'none' if collisions.empty else '')
if not collisions.empty:
    print(collisions.to_string(index=False))

print('\nFiles per group and condition:')
print(inventory.pivot_table(index='group', columns='condition', values='file',
                            aggfunc='count', fill_value=0))

## 2. Acquisition parameters and recording timestamps

Sampling rate, hardware filters, duration and start time are read from each EDF header.

In [ ]:
header_rows = []
for fp in sorted(Path(DATA_DIR).glob('*.edf')):
    raw = mne.io.read_raw_edf(fp, preload=False, verbose=False)
    start = raw.info['meas_date']
    duration = raw.n_times / raw.info['sfreq']
    header_rows.append({
        'file': fp.name,
        'sfreq': raw.info['sfreq'],
        'highpass_hz': raw.info['highpass'],
        'lowpass_hz': raw.info['lowpass'],
        'n_channels': len(raw.ch_names),
        'missing_eeg_channels': ';'.join(ch for ch in EEG_CHANNELS if ch not in raw.ch_names),
        'duration_s': round(duration, 1),
        'start': start,
        'end': start + timedelta(seconds=duration) if start is not None else None,
    })
inventory = inventory.merge(pd.DataFrame(header_rows), on='file')
inventory.to_csv(os.path.join(OUTPUT_DIR, 'file_inventory.csv'), index=False)

print('Distinct acquisition parameters per group:')
for col in ['sfreq', 'highpass_hz', 'lowpass_hz']:
    print(f'  {col:12s}', inventory.groupby('group')[col].unique().to_dict())

missing = inventory[inventory['missing_eeg_channels'] != '']
print('\nFiles missing EEG channels:', 'none' if missing.empty else '')
if not missing.empty:
    print(missing[['file', 'missing_eeg_channels']].to_string(index=False))

print('\nRecording dates per group and condition:')
inventory['date'] = pd.to_datetime(inventory['start']).dt.date
print(inventory.groupby(['group', 'condition'])['date'].agg(['min', 'max']))

## 3. Two-level SHA-256 audit

- **File level:** SHA-256 of the complete EDF file. Equal hashes mean byte-identical files.
- **Signal level:** SHA-256 of the 19 EEG channels only, in µV rounded to three decimals,
  independent of header and metadata. Equal hashes with different file-level hashes mean
  the same physiological recording stored with a different header (for example, a
  different subject identifier).

In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def sha256_signal(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw.pick([ch for ch in EEG_CHANNELS if ch in raw.ch_names])
    x = np.round(raw.get_data() * 1e6, 3)          # µV, 3 decimals
    return hashlib.sha256(x.astype('<f8').tobytes()).hexdigest()


audit = inventory.copy()
audit['sha256_file'] = [sha256_file(os.path.join(DATA_DIR, f)) for f in audit['file']]
audit['sha256_signal'] = [sha256_signal(os.path.join(DATA_DIR, f)) for f in audit['file']]

for level in ['sha256_file', 'sha256_signal']:
    counts = audit[level].map(audit[level].value_counts())
    audit[f'{level}_group_size'] = counts
    dup_groups = audit[counts > 1].groupby(level)['file'].apply(list)
    print(f'\n{level}: {len(dup_groups)} group(s) of identical records')
    for files in dup_groups:
        print('   ' + ' = '.join(files))

audit.to_csv(os.path.join(OUTPUT_DIR, 'duplicate_audit.csv'), index=False)

## 4. Attribution of shared records from recording timestamps

For each record whose signal appears under more than one identifier, the start time of
the shared record is compared with the EC and EO records of every candidate subject.
The protocol order is EC, EO and TASK, so a TASK record that starts shortly after the
end of a candidate's EO record, on the same day, is attributed to that candidate.

In [ ]:
shared = audit[audit['sha256_signal_group_size'] > 1]
attribution_rows = []

for signal_hash, grp in shared.groupby('sha256_signal'):
    record_start = grp['start'].iloc[0]
    condition = grp['condition'].iloc[0]
    for _, rec in grp.iterrows():
        own = audit[(audit['subject_id'] == rec['subject_id']) &
                    (audit['condition'] != condition)]
        row = {'shared_signal': signal_hash[:12], 'condition': condition,
               'file': rec['file'], 'subject_id': rec['subject_id'],
               'shared_record_start': record_start}
        for c in CONDITIONS:
            other = own[own['condition'] == c]
            if len(other):
                row[f'{c}_start'] = other['start'].iloc[0]
                row[f'{c}_end'] = other['end'].iloc[0]
        # Gap between the end of the subject's preceding record and the shared record.
        preceding = {'TASK': 'EO', 'EO': 'EC'}.get(condition)
        if preceding and f'{preceding}_end' in row:
            row['gap_after_preceding_s'] = (record_start - row[f'{preceding}_end']).total_seconds()
        attribution_rows.append(row)

attribution = pd.DataFrame(attribution_rows)
if 'gap_after_preceding_s' not in attribution:
    attribution['gap_after_preceding_s'] = np.nan
if attribution.empty:
    print('No shared records.')
else:
    attribution.to_csv(os.path.join(OUTPUT_DIR, 'shared_record_attribution.csv'), index=False)
    # Within each shared record, the candidate with the smallest gap is listed first.
    order = attribution.assign(abs_gap=attribution['gap_after_preceding_s'].abs())
    order = order.sort_values(['shared_signal', 'abs_gap']).drop(columns='abs_gap')
    with pd.option_context('display.width', 200, 'display.max_columns', 20):
        print(order.to_string(index=False))

## 5. Exclusion decisions

Decisions taken from the audit of the original dataset (version 2):

| Records | Finding | Decision |
|---|---|---|
| HC 30 — EC, EO, TASK | Signal identical to HC 27 in all three conditions | Exclude HC 30 |
| MDD 34 — EC, EO, TASK | Signal identical to MDD 33 in all three conditions | Exclude MDD 34 |
| TASK of HC 1, 2, 3, 4, 7, 10, 11, 12, 17, 29 | One signal stored under ten identifiers | Keep it as HC 4; exclude the other nine copies |

The shared TASK record starts on 2013-01-12 at 16:23:35. The EC and EO records of HC 4
were made the same day, starting at 16:11:55 (301 s) and 16:18:00 (295 s); the TASK record
starts 40 s after the end of the EO record of HC 4, in the EC, EO, TASK order of the
protocol. Because this attribution relies on metadata, the main analyses are repeated
without HC 4 (sensitivity analysis).

The cell below writes the exclusion list and checks that every excluded record has an
identical signal in a record that is kept.

In [ ]:
EXCLUDED_RECORDS = (
    [('HC', 30, c, 'signal identical to HC 27') for c in CONDITIONS]
    + [('MDD', 34, c, 'signal identical to MDD 33') for c in CONDITIONS]
    + [('HC', s, 'TASK', 'copy of the shared TASK record attributed to HC 4')
       for s in (1, 2, 3, 7, 10, 11, 12, 17, 29)]
)
excluded = pd.DataFrame(EXCLUDED_RECORDS, columns=['group', 'subject', 'condition', 'reason'])
excluded.to_csv(os.path.join(OUTPUT_DIR, 'excluded_records.csv'), index=False)

key = ['group', 'subject', 'condition']
flagged = audit.merge(excluded[key], on=key, how='left', indicator=True)
audit['excluded'] = (flagged['_merge'] == 'both').values
kept_hashes = set(audit.loc[~audit['excluded'], 'sha256_signal'])

not_found = excluded.merge(audit[key], on=key, how='left', indicator=True)
not_found = not_found[not_found['_merge'] == 'left_only']
if len(not_found):
    print('Excluded records not present in DATA_DIR:')
    print(not_found[key].to_string(index=False))

unmatched = audit[audit['excluded'] & ~audit['sha256_signal'].isin(kept_hashes)]
assert unmatched.empty, f'Excluded records without an identical kept record:\n{unmatched[key]}'
remaining = audit[~audit['excluded']]
assert remaining['sha256_signal'].is_unique, 'Duplicate signals remain after exclusion.'
print('Check passed: every excluded record has an identical kept record; no duplicates remain.')

summary = pd.DataFrame({
    'original': audit.groupby(['group', 'condition']).size(),
    'after_duplicate_removal': remaining.groupby(['group', 'condition']).size(),
})
print('\nRecords per group and condition (Table 1):')
print(summary)